# TPR Experiment for Adaptive CoRT-SI

This notebook estimates a per-run true positive rate (TPR) under the alternative model. Each run generates one synthetic dataset, computes selective p-values with `SI(...)`, samples 20 random true-signal p-values with replacement, and records the rejection rate at a fixed level $\alpha = 0.05$.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
REPO_ROOT = next((candidate for candidate in candidates if (candidate / "cort_si").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the repo root containing the cort_si package.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from cort_si import SI, gen_data

RESULTS_DIR = REPO_ROOT / "run_experiment" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
p = 300
nS = 100
nT = 50
K = 2
lambda_sel = 0.5
lambda0 = 0.5
T = 3

num_runs = 100
num_random_j = 20
alpha = 0.05
seed = 0

true_beta_scale = 1.0
rho = 0.0
sigma_noise = 1.0
source_shift_sd = 0.3
covariate_shift = False

print(f"num_runs = {num_runs}")
print(f"num_random_j per run = {num_random_j}")
print(f"alpha = {alpha}")
print(f"K = {K}, p = {p}, nS = {nS}, nT = {nT}")

In [ ]:
def generate_alternative_instance(iteration_seed):
    return gen_data.generate_data(
        p=p,
        nS=nS,
        nT=nT,
        K=K,
        true_beta=true_beta_scale,
        rho=rho,
        sigma_noise=sigma_noise,
        source_shift_sd=source_shift_sd,
        covariate_shift=covariate_shift,
        seed=iteration_seed,
    )


def extract_true_signal_pvalues(pvalue_pairs, true_support):
    if pvalue_pairs is None:
        return np.array([], dtype=float)

    filtered = [
        pvalue
        for feature_idx, pvalue in pvalue_pairs
        if pvalue is not None and np.isfinite(pvalue) and feature_idx in true_support
    ]
    return np.asarray(filtered, dtype=float)


def sample_true_signal_pvalues(true_pvalues, num_samples, rng):
    if true_pvalues.size == 0:
        return np.full(num_samples, np.nan)

    return rng.choice(true_pvalues, size=num_samples, replace=True)


def run_tpr_experiment(num_runs, num_random_j, alpha, seed):
    rng = np.random.default_rng(seed)
    tpr_values = []
    sampled_pvalues = []
    selected_true_counts = []
    selected_total_counts = []

    for iteration in range(num_runs):
        data_seed = seed + iteration
        XS_list, YS_list, X0, Y0, _, SigmaS_list, Sigma0, beta_0 = generate_alternative_instance(data_seed)
        lambdak_list = [lambda0] * len(XS_list)
        true_support = set(np.flatnonzero(beta_0))

        pvalue_pairs = SI(
            X0,
            Y0,
            XS_list,
            YS_list,
            lambda_sel=lambda_sel,
            lambda0=lambda0,
            lambdak_list=lambdak_list,
            SigmaS_list=SigmaS_list,
            Sigma0=Sigma0,
            T=T,
            verbose=False,
        )

        true_pvalues = extract_true_signal_pvalues(pvalue_pairs, true_support)
        sampled = sample_true_signal_pvalues(true_pvalues, num_random_j, rng)
        tpr = float(np.mean(sampled <= alpha)) if np.isfinite(sampled).any() else 0.0

        tpr_values.append(tpr)
        sampled_pvalues.append(sampled)
        selected_true_counts.append(int(true_pvalues.size))
        selected_total_counts.append(0 if pvalue_pairs is None else len(pvalue_pairs))

        if (iteration + 1) % 10 == 0:
            print(f"Completed {iteration + 1}/{num_runs} runs")

    return {
        "tpr_values": np.asarray(tpr_values, dtype=float),
        "sampled_pvalues": np.asarray(sampled_pvalues, dtype=float),
        "selected_true_counts": np.asarray(selected_true_counts, dtype=int),
        "selected_total_counts": np.asarray(selected_total_counts, dtype=int),
    }

In [ ]:
tpr_results = run_tpr_experiment(
    num_runs=num_runs,
    num_random_j=num_random_j,
    alpha=alpha,
    seed=seed,
)

tpr_path = RESULTS_DIR / "tpr_results.npz"
np.savez(
    tpr_path,
    tpr_values=tpr_results["tpr_values"],
    sampled_pvalues=tpr_results["sampled_pvalues"],
    selected_true_counts=tpr_results["selected_true_counts"],
    selected_total_counts=tpr_results["selected_total_counts"],
    alpha=np.asarray([alpha], dtype=float),
    num_runs=np.asarray([num_runs], dtype=int),
    num_random_j=np.asarray([num_random_j], dtype=int),
)

print(f"Saved TPR results to {tpr_path}")
print(f"Mean TPR@{alpha:.2f}: {tpr_results['tpr_values'].mean():.3f}")
print(f"Median TPR@{alpha:.2f}: {np.median(tpr_results['tpr_values']):.3f}")
print(f"Runs with at least one selected true signal: {(tpr_results['selected_true_counts'] > 0).sum()}/{num_runs}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.boxplot(
    tpr_results['tpr_values'],
    patch_artist=True,
    boxprops={"facecolor": "#b8d8ba", "edgecolor": "#2f5d50"},
    medianprops={"color": "#b22222", "linewidth": 2},
    whiskerprops={"color": "#2f5d50"},
    capprops={"color": "#2f5d50"},
)
jitter_rng = np.random.default_rng(seed)
x_jitter = 1.0 + 0.06 * jitter_rng.normal(size=tpr_results['tpr_values'].size)
ax.scatter(x_jitter, tpr_results['tpr_values'], alpha=0.55, color="#205072", s=24)
ax.set_xticks([1])
ax.set_xticklabels([f"TPR @ alpha={alpha:.2f}"])
ax.set_ylabel("Rejection rate across 20 sampled true signals")
ax.set_title("Adaptive CoRT-SI TPR over 100 runs")
ax.set_ylim(-0.02, 1.02)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()